<a href="https://colab.research.google.com/github/AdriaLazaro/btc-cycle-model/blob/main/notebooks/BTC_Cycle_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BTC Cycle Investment Model

Notebook limpio y modular para construir el oscilador de ciclos de Bitcoin.


## Resumen ejecutivo

Con el dataset actual, el ultimo cierre disponible es **2026-06-16** a aproximadamente **$65,638**. La fase actual del oscilador es **0.05**, muy cerca de la linea de tendencia, dentro de una lectura de enfriamiento post-pico 2025.

Con la fuente unica de parametros (`DRAWDOWN_MEAN/STD` y `MULT_MEAN/STD`), el rango parametrico del proximo minimo queda en **$29,646-$41,998** entre **2026-09-26** y **2026-11-18**. El rango parametrico del maximo posterior queda en **$91,902-$222,590** entre **2029-08-16** y **2029-10-16**.

El ensemble final usa **3,000 escenarios** y valida que los percentiles principales no sean degenerados. Salvedades clave: solo hay 3-4 ciclos historicos comparables, los turning points se definen manualmente por ventanas, y el ensemble es una herramienta de decision bajo incertidumbre, no asesoramiento financiero ni una prediccion determinista.


## Nota metodologica: oscilador amortiguado y circularidad

La idea de visualizar Bitcoin como un ciclo amortiguado es util: comunica de forma clara que los picos y valles historicos han sido cada vez menos extremos. Como herramienta visual y de decision, esa intuicion tiene valor.

Pero hay una limitacion importante. Las fases de los turning points se asignan manualmente en `build_turning_points()` (por ejemplo, Max 2013 = 1.00, Max 2017 = 0.75, Max 2021 = 0.50, Max 2025 = 0.35). Por tanto, calcular despues `beta_peak` o `beta_trough` a partir de esas fases confirma una amortiguacion que ya fue impuesta cualitativamente. No es una estimacion estadistica independiente aprendida del precio.

En este notebook, las betas se tratan como parametros descriptivos para construir envolventes razonables, no como inferencia estadistica robusta. El ensemble futuro usa anclas de precio/fecha y Monte Carlo para explorar escenarios plausibles, pero no pretende ser una prediccion puntual ni un modelo de machine learning entrenado sobre datos brutos.


## Imports y configuracion

Aqui se concentran los parametros editables del modelo. La parte importante es que drawdown y multiplicador tienen una sola fuente de verdad: los rangos usados en el grafico y en el ensemble derivan de los mismos parametros.

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path


# Dataset y rango historico principal.
DATASET_CANDIDATES = [
    Path("coin-metrics-new-chart.tsv.tsv"),
    Path("../coin-metrics-new-chart.tsv.tsv"),
    Path("/content/coin-metrics-new-chart.tsv.tsv"),
    Path("/content/BTC.tsv.tsv"),
    Path("/content/btc-cycle-model/coin-metrics-new-chart.tsv.tsv"),
    Path("/content/btc-cycle-model-main/coin-metrics-new-chart.tsv.tsv"),
]

START_DATE = "2013-01-01"
FUTURE_END = pd.Timestamp("2031-03-01")

try:
    from IPython.display import display
except ImportError:
    display = print


# ============================================================
# Fuente unica de parametros de drawdown y multiplicador
# ============================================================
# Estos parametros alimentan tanto el grafico historico como el ensemble.
# Asi evitamos que dos secciones del notebook muestren rangos incompatibles.
DRAWDOWN_MEAN = 0.71
DRAWDOWN_STD = 0.05
DRAWDOWN_MIN = 0.55
DRAWDOWN_MAX = 0.85

MULT_MEAN = 4.2
MULT_STD = 1.1
MULT_MIN_CLIP = 2.2
MULT_MAX_CLIP = 7.5

DRAWDOWN_SUAVE = max(DRAWDOWN_MIN, DRAWDOWN_MEAN - DRAWDOWN_STD)
DRAWDOWN_DURO = min(DRAWDOWN_MAX, DRAWDOWN_MEAN + DRAWDOWN_STD)
MULT_MIN = max(MULT_MIN_CLIP, MULT_MEAN - MULT_STD)
MULT_MAX = min(MULT_MAX_CLIP, MULT_MEAN + MULT_STD)

# Suavizado semanal del oscilador: 1 conserva todo el detalle real.
ROLLING_SMOOTH = 1


# ============================================================
# Fase 3: ensemble probabilistico de escenarios futuros
# ============================================================
N_MODELS = 3000
N_PLOT = 300
HORIZON_DAYS = 1300
RANDOM_SEED = 42

ENTRY_PRICES = np.arange(25000, 70001, 2500)
TARGET_PRICES = [100000, 150000, 200000, 250000]

MAX_REASONABLE_PRICE = 450000
MIN_REASONABLE_PRICE = 10000

HALVING_DATES = [
    "2012-11-28",
    "2016-07-09",
    "2020-05-11",
    "2024-04-20",
]
NEXT_HALVING_ESTIMATE = "2028-04-15"

BACKTEST_N_MODELS = 1200
CALIBRATION_BINS = np.linspace(0.0, 1.0, 6)


# Fase 3.1: realismo visual/dinamico del ensemble
SHOW_ENSEMBLE_BANDS = False
SHOW_ENSEMBLE_MEDIAN = True
ENSEMBLE_PATH_ALPHA = 0.045
ENSEMBLE_PATH_LINEWIDTH = 0.8

PHASE_NOISE_SCALE_MIN = 0.035
PHASE_NOISE_SCALE_MAX = 0.10
PHASE_NOISE_RHO_MIN = 0.75
PHASE_NOISE_RHO_MAX = 0.94

PRICE_NOISE_SCALE_MIN = 0.03
PRICE_NOISE_SCALE_MAX = 0.085

ENVELOPE_MARGIN_PHASE = 0.06

SHOW_ENSEMBLE_EXTREMA_LABELS = True
ENSEMBLE_EXTREMA_USE = "median"  # opciones: "median" o "mean"


## Funciones reutilizables

Estas funciones cargan el dataset, preparan la serie semanal, definen turning points historicos mediante ventanas manuales y construyen el oscilador. Los turning points no salen de un algoritmo automatico: son anclas cualitativas elegidas para representar los ciclos historicos.

In [ ]:
def year_diff(d2, d1):
    """Return the distance between two dates in years."""
    return (pd.Timestamp(d2) - pd.Timestamp(d1)).days / 365.25


def load_btc_data(candidate_paths=None, encodings=("utf-8", "utf-16")):
    """Load BTC price data from several local/Colab paths and print a quick audit."""
    candidate_paths = candidate_paths or DATASET_CANDIDATES
    required_columns = {"Time", "BTC / USD Denominated Closing Price"}
    load_errors = []

    for path in candidate_paths:
        path = Path(path)
        if not path.exists():
            continue

        for encoding in encodings:
            try:
                raw_df = pd.read_csv(path, sep="\t", encoding=encoding)
                missing = required_columns.difference(raw_df.columns)
                if missing:
                    raise ValueError(f"Missing columns: {sorted(missing)}")

                dates = pd.to_datetime(raw_df["Time"], errors="coerce")
                print(f"Dataset path: {path.resolve()}")
                print(f"Encoding: {encoding}")
                print(f"Rows: {len(raw_df):,}")
                print(f"Dates: {dates.min().date()} -> {dates.max().date()}")
                print(f"Columns: {list(raw_df.columns)}")
                return raw_df
            except Exception as exc:
                load_errors.append(f"{path} ({encoding}): {exc}")

    searched = "\n".join(str(path) for path in candidate_paths)
    details = "\n".join(load_errors)
    raise FileNotFoundError(
        f"Dataset not found. Searched:\n{searched}\n\nLoad errors:\n{details}"
    )


def prepare_btc_data(raw_df, start_date=START_DATE):
    """Clean raw BTC data and build the weekly log-price dataset used by the model."""
    df = raw_df.rename(columns={
        "Time": "date",
        "BTC / USD Denominated Closing Price": "price",
    }).copy()

    df["date"] = pd.to_datetime(df["date"])
    df["price"] = pd.to_numeric(df["price"], errors="coerce")
    df = df.dropna(subset=["date", "price"])
    df = df.sort_values("date")

    btc = df[df["date"] >= start_date].copy()
    weekly = (
        btc.set_index("date")
           .resample("W")
           .last()
           .dropna()
           .reset_index()
    )
    weekly["log_price"] = np.log(weekly["price"])

    return df, btc, weekly


def get_turning_point(weekly, start, end, kind, label, phase):
    """Find the real weekly max/min inside a manual cycle window."""
    subset = weekly[(weekly["date"] >= start) & (weekly["date"] <= end)].copy()
    if len(subset) == 0:
        raise ValueError(f"No data between {start} and {end}")

    if kind == "max":
        row = subset.loc[subset["price"].idxmax()]
    else:
        row = subset.loc[subset["price"].idxmin()]

    return {
        "date": row["date"],
        "price": row["price"],
        "log_price": np.log(row["price"]),
        "label": label,
        "phase": phase,
        "kind": kind,
    }


def build_turning_points(weekly):
    """Build the manual cycle anchors used to map log-price into cycle phase."""
    windows = [
        ("2013-01-01", "2013-04-30", "min", "Inicio 2013", -0.35),
        ("2013-10-01", "2014-01-31", "max", "Max 2013", 1.00),
        ("2014-12-01", "2015-03-31", "min", "Min 2015", -0.75),
        ("2017-11-01", "2018-01-31", "max", "Max 2017", 0.75),
        ("2018-11-01", "2019-02-28", "min", "Min 2018", -0.50),
        ("2021-10-01", "2021-12-31", "max", "Max 2021", 0.50),
        ("2022-10-01", "2023-01-31", "min", "Min 2022", -0.35),
        ("2025-01-01", "2026-03-01", "max", "Max 2025", 0.35),
    ]

    turns = [
        get_turning_point(weekly, start, end, kind, label, phase)
        for start, end, kind, label, phase in windows
    ]

    return pd.DataFrame(turns).sort_values("date").reset_index(drop=True)


def compute_damping_parameters(turning_points):
    """Estimate exponential damping bands from historical peak/trough amplitudes."""
    peaks = turning_points[turning_points["kind"] == "max"].copy()
    troughs = turning_points[
        (turning_points["kind"] == "min") &
        (turning_points["label"] != "Inicio 2013")
    ].copy()

    peak_dates = peaks["date"].to_list()
    peak_amps = peaks["phase"].abs().to_numpy()
    trough_dates = troughs["date"].to_list()
    trough_amps = troughs["phase"].abs().to_numpy()

    peak_betas = []
    for i in range(len(peak_amps) - 1):
        dt = year_diff(peak_dates[i + 1], peak_dates[i])
        peak_betas.append(-np.log(peak_amps[i + 1] / peak_amps[i]) / dt)

    trough_betas = []
    for i in range(len(trough_amps) - 1):
        dt = year_diff(trough_dates[i + 1], trough_dates[i])
        trough_betas.append(-np.log(trough_amps[i + 1] / trough_amps[i]) / dt)

    peak_betas = np.array(peak_betas)
    trough_betas = np.array(trough_betas)

    beta_peak_mean = peak_betas.mean()
    beta_peak_std = peak_betas.std(ddof=1)
    beta_trough_mean = trough_betas.mean()
    beta_trough_std = trough_betas.std(ddof=1)

    return {
        "peaks": peaks,
        "troughs": troughs,
        "peak_betas": peak_betas,
        "trough_betas": trough_betas,
        "beta_peak_mean": beta_peak_mean,
        "beta_peak_std": beta_peak_std,
        "beta_peak_low": max(0.001, beta_peak_mean - beta_peak_std),
        "beta_peak_high": beta_peak_mean + beta_peak_std,
        "beta_trough_mean": beta_trough_mean,
        "beta_trough_std": beta_trough_std,
        "beta_trough_low": max(0.001, beta_trough_mean - beta_trough_std),
        "beta_trough_high": beta_trough_mean + beta_trough_std,
    }


def estimate_next_cycle_windows(
    turning_points,
    damping,
    drawdown_suave=DRAWDOWN_SUAVE,
    drawdown_duro=DRAWDOWN_DURO,
    mult_min=MULT_MIN,
    mult_max=MULT_MAX,
):
    """Estimate next trough/peak timing windows and price ranges."""
    peaks = damping["peaks"]
    troughs = damping["troughs"]
    peak_dates = peaks["date"].to_list()

    peak_to_peak = np.array([
        year_diff(peak_dates[i + 1], peak_dates[i])
        for i in range(len(peak_dates) - 1)
    ])

    historical_peak_to_trough = np.array([
        year_diff(troughs.iloc[0]["date"], peaks.iloc[0]["date"]),
        year_diff(troughs.iloc[1]["date"], peaks.iloc[1]["date"]),
        year_diff(troughs.iloc[2]["date"], peaks.iloc[2]["date"]),
    ])

    pp_mean = peak_to_peak.mean()
    pp_std = peak_to_peak.std(ddof=1)
    pt_mean = historical_peak_to_trough.mean()
    pt_std = historical_peak_to_trough.std(ddof=1)

    last_peak = peaks.iloc[-1]
    last_peak_date = last_peak["date"]
    last_peak_price = last_peak["price"]

    next_min_early = last_peak_date + pd.to_timedelta((pt_mean - pt_std) * 365.25, unit="D")
    next_min_late = last_peak_date + pd.to_timedelta((pt_mean + pt_std) * 365.25, unit="D")
    next_min_center = last_peak_date + pd.to_timedelta(pt_mean * 365.25, unit="D")

    next_max_early = last_peak_date + pd.to_timedelta((pp_mean - pp_std) * 365.25, unit="D")
    next_max_late = last_peak_date + pd.to_timedelta((pp_mean + pp_std) * 365.25, unit="D")
    next_max_center = last_peak_date + pd.to_timedelta(pp_mean * 365.25, unit="D")

    min_price_low = last_peak_price * (1 - drawdown_duro)
    min_price_high = last_peak_price * (1 - drawdown_suave)
    max_price_low = min_price_low * mult_min
    max_price_high = min_price_high * mult_max

    last_trough = troughs.iloc[-1]
    dt_next_trough = year_diff(next_min_center, last_trough["date"])

    next_trough_amp_less_damp = abs(last_trough["phase"]) * np.exp(
        -damping["beta_trough_low"] * dt_next_trough
    )
    next_trough_amp_more_damp = abs(last_trough["phase"]) * np.exp(
        -damping["beta_trough_high"] * dt_next_trough
    )
    next_trough_phase = -np.mean([
        next_trough_amp_less_damp,
        next_trough_amp_more_damp,
    ])

    projected_min = {
        "date": next_min_center,
        "price": (min_price_low + min_price_high) / 2,
        "log_price": np.log((min_price_low + min_price_high) / 2),
        "label": "Min proyectado",
        "phase": next_trough_phase,
        "kind": "min",
    }

    turns_for_mapping = pd.concat([
        turning_points,
        pd.DataFrame([projected_min]),
    ]).sort_values("date").reset_index(drop=True)

    return {
        "peak_to_peak": peak_to_peak,
        "historical_peak_to_trough": historical_peak_to_trough,
        "pp_mean": pp_mean,
        "pp_std": pp_std,
        "pt_mean": pt_mean,
        "pt_std": pt_std,
        "last_peak": last_peak,
        "next_min_early": next_min_early,
        "next_min_late": next_min_late,
        "next_min_center": next_min_center,
        "next_max_early": next_max_early,
        "next_max_late": next_max_late,
        "next_max_center": next_max_center,
        "min_price_low": min_price_low,
        "min_price_high": min_price_high,
        "max_price_low": max_price_low,
        "max_price_high": max_price_high,
        "projected_min": projected_min,
        "turns_for_mapping": turns_for_mapping,
    }


def build_cycle_oscillator(weekly, turns_for_mapping, rolling_smooth=ROLLING_SMOOTH):
    """Map weekly log-price into cycle phase, normalized segment by segment."""
    segments = []

    for i in range(len(turns_for_mapping) - 1):
        start = turns_for_mapping.iloc[i]
        end = turns_for_mapping.iloc[i + 1]
        seg_end_date = min(end["date"], weekly["date"].max())

        seg = weekly[
            (weekly["date"] >= start["date"]) &
            (weekly["date"] <= seg_end_date)
        ].copy()

        if len(seg) == 0:
            continue

        log_start = start["log_price"]
        log_end = end["log_price"]
        phase_start = start["phase"]
        phase_end = end["phase"]

        if abs(log_end - log_start) < 1e-9:
            continue

        seg["ratio"] = (seg["log_price"] - log_start) / (log_end - log_start)
        seg["cycle_raw"] = phase_start + seg["ratio"] * (phase_end - phase_start)

        low = min(phase_start, phase_end) - 0.12
        high = max(phase_start, phase_end) + 0.12
        seg["cycle_raw"] = seg["cycle_raw"].clip(low, high)

        segments.append(seg)

    cycle = pd.concat(segments).sort_values("date").drop_duplicates("date")
    cycle["cycle_smooth"] = (
        cycle["cycle_raw"]
        .rolling(rolling_smooth, center=True, min_periods=1)
        .mean()
    )

    return cycle


def compute_envelopes(damping, future_end=FUTURE_END, n_points=700):
    """Compute upper/lower exponential damping envelopes for the oscillator."""
    peaks = damping["peaks"]
    troughs = damping["troughs"]

    env_dates = pd.date_range(peaks.iloc[0]["date"], future_end, periods=n_points)

    upper_anchor_date = peaks.iloc[0]["date"]
    upper_anchor_amp = abs(peaks.iloc[0]["phase"])
    upper_t = np.array([
        max(0, year_diff(date, upper_anchor_date))
        for date in env_dates
    ])
    upper_less_damp = upper_anchor_amp * np.exp(-damping["beta_peak_low"] * upper_t)
    upper_more_damp = upper_anchor_amp * np.exp(-damping["beta_peak_high"] * upper_t)

    lower_anchor_date = troughs.iloc[0]["date"]
    lower_anchor_amp = abs(troughs.iloc[0]["phase"])
    lower_t = np.array([
        max(0, year_diff(date, lower_anchor_date))
        for date in env_dates
    ])
    lower_less_damp = -lower_anchor_amp * np.exp(-damping["beta_trough_low"] * lower_t)
    lower_more_damp = -lower_anchor_amp * np.exp(-damping["beta_trough_high"] * lower_t)

    return {
        "env_dates": env_dates,
        "upper_less_damp": upper_less_damp,
        "upper_more_damp": upper_more_damp,
        "lower_less_damp": lower_less_damp,
        "lower_more_damp": lower_more_damp,
    }


def plot_cycle_oscillator(
    weekly,
    cycle,
    turning_points,
    envelopes,
    windows,
    future_end=FUTURE_END,
):
    """Plot the BTC cycle oscillator with cycle backgrounds, envelopes, and windows."""
    fig, ax = plt.subplots(figsize=(18, 8))

    cycle_spans = [
        (pd.Timestamp("2013-11-01"), pd.Timestamp("2017-12-31"), "#eaf4ff", "Ciclo 2013-2017"),
        (pd.Timestamp("2017-12-31"), pd.Timestamp("2021-11-30"), "#fff3e6", "Ciclo 2017-2021"),
        (pd.Timestamp("2021-11-30"), pd.Timestamp("2025-12-31"), "#edf8ee", "Ciclo 2021-2025"),
        (pd.Timestamp("2025-12-31"), future_end, "#f5edff", "Ciclo proyectado 2025-2029"),
    ]

    for start, end, color, _ in cycle_spans:
        ax.axvspan(start, end, color=color, alpha=0.45, zorder=0)

    ax.plot(
        cycle["date"],
        cycle["cycle_smooth"],
        linewidth=2.8,
        color="#3f73b5",
        label="Ciclo BTC normalizado con datos reales",
        zorder=4,
    )
    ax.plot(
        cycle["date"],
        cycle["cycle_raw"],
        linewidth=0.7,
        alpha=0.25,
        color="#f2a65a",
        label="Detalle semanal real",
        zorder=3,
    )
    ax.axhline(
        0,
        linewidth=2.8,
        color="#3f73b5",
        alpha=0.9,
        label="Tendencia normalizada",
        zorder=2,
    )

    env_dates = envelopes["env_dates"]
    ax.fill_between(
        env_dates,
        envelopes["upper_more_damp"],
        envelopes["upper_less_damp"],
        color="#ff7b72",
        alpha=0.16,
        label="Envolvente superior +/- amortiguamiento",
        zorder=1,
    )
    ax.plot(env_dates, envelopes["upper_less_damp"], "--", color="#ff7b72", linewidth=2, zorder=5)
    ax.plot(env_dates, envelopes["upper_more_damp"], "--", color="#ff7b72", linewidth=2, zorder=5)

    ax.fill_between(
        env_dates,
        envelopes["lower_less_damp"],
        envelopes["lower_more_damp"],
        color="#2fbf71",
        alpha=0.16,
        label="Envolvente inferior +/- amortiguamiento",
        zorder=1,
    )
    ax.plot(env_dates, envelopes["lower_less_damp"], "--", color="#2fbf71", linewidth=2, zorder=5)
    ax.plot(env_dates, envelopes["lower_more_damp"], "--", color="#2fbf71", linewidth=2, zorder=5)

    ax.axvspan(
        windows["next_min_early"],
        windows["next_min_late"],
        color="#f6bd60",
        alpha=0.22,
        label="Ventana proximo minimo",
        zorder=1,
    )
    ax.axvline(windows["next_min_center"], color="#f6bd60", linestyle=":", linewidth=3, zorder=6)

    ax.axvspan(
        windows["next_max_early"],
        windows["next_max_late"],
        color="#c77dff",
        alpha=0.18,
        label="Ventana proximo maximo",
        zorder=1,
    )
    ax.axvline(windows["next_max_center"], color="#c77dff", linestyle=":", linewidth=3, zorder=6)

    ax.scatter(
        turning_points["date"],
        turning_points["phase"],
        s=70,
        color="#3f73b5",
        zorder=7,
    )

    for _, row in turning_points.iterrows():
        text = f"{row['label']}\n${row['price']:,.0f}"
        offset = 12 if row["phase"] >= 0 else -28
        va = "bottom" if row["phase"] >= 0 else "top"
        ax.annotate(
            text,
            (row["date"], row["phase"]),
            textcoords="offset points",
            xytext=(0, offset),
            ha="center",
            va=va,
            fontsize=8.5,
            zorder=8,
        )

    last = weekly.iloc[-1]
    last_cycle = cycle.iloc[-1]
    ax.scatter(
        last_cycle["date"],
        last_cycle["cycle_smooth"],
        s=90,
        color="#f28e2b",
        zorder=8,
    )
    ax.annotate(
        f"Hoy\n${last['price']:,.0f}",
        (last_cycle["date"], last_cycle["cycle_smooth"]),
        textcoords="offset points",
        xytext=(12, -8),
        ha="left",
        va="center",
        fontsize=9,
        zorder=9,
    )

    min_text = (
        "Proximo minimo\n"
        f"{windows['next_min_early'].strftime('%b %Y')} - {windows['next_min_late'].strftime('%b %Y')}\n"
        f"${windows['min_price_low']:,.0f} - ${windows['min_price_high']:,.0f}"
    )
    max_text = (
        "Proximo maximo\n"
        f"{windows['next_max_early'].strftime('%b %Y')} - {windows['next_max_late'].strftime('%b %Y')}\n"
        f"${windows['max_price_low']:,.0f} - ${windows['max_price_high']:,.0f}"
    )

    ax.text(
        windows["next_min_center"],
        -0.92,
        min_text,
        ha="center",
        va="top",
        fontsize=10,
        bbox=dict(boxstyle="round,pad=0.35", facecolor="white", edgecolor="#f6bd60", alpha=0.96),
        zorder=10,
    )
    ax.text(
        windows["next_max_center"],
        0.82,
        max_text,
        ha="center",
        va="bottom",
        fontsize=10,
        bbox=dict(boxstyle="round,pad=0.35", facecolor="white", edgecolor="#c77dff", alpha=0.96),
        zorder=10,
    )

    cycle_labels = [
        (pd.Timestamp("2015-12-01"), "Ciclo 2013-2017"),
        (pd.Timestamp("2019-11-01"), "Ciclo 2017-2021"),
        (pd.Timestamp("2023-11-01"), "Ciclo 2021-2025"),
        (pd.Timestamp("2028-02-01"), "Ciclo proyectado"),
    ]
    for x_pos, label in cycle_labels:
        ax.text(x_pos, -1.08, label, ha="center", va="bottom", fontsize=9, alpha=0.65)

    ax.set_yticks([-1.0, -0.5, 0.0, 0.5, 1.0])
    ax.set_yticklabels([
        "Capitulacion",
        "Infravalorado",
        "Tendencia",
        "Sobrecalentado",
        "Euforia",
    ])
    ax.set_ylim(-1.15, 1.15)
    ax.set_xlim(pd.Timestamp("2013-01-01"), future_end)
    ax.set_title(
        "Bitcoin: oscilador de ciclos desde 2013 con detalle semanal real\n"
        "y envolventes exponenciales de amortiguamiento",
        fontsize=15,
    )
    ax.set_xlabel("Fecha")
    ax.set_ylabel("Fase del ciclo")
    ax.grid(True, axis="y", alpha=0.3)
    ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1.0), fontsize=9, frameon=True)

    plt.tight_layout(rect=[0, 0, 0.82, 1])
    return fig, ax


## Ejecutar modelo y grafico principal

Esta fase construye el oscilador historico y sus envolventes. Sirve para explicar donde esta el ciclo actual en terminos de fase: capitulacion, infravaloracion, tendencia, sobrecalentamiento o euforia.

In [ ]:
raw_df = load_btc_data()
df, btc, weekly = prepare_btc_data(raw_df)

turns_actual = build_turning_points(weekly)
damping = compute_damping_parameters(turns_actual)
windows = estimate_next_cycle_windows(turns_actual, damping)
cycle = build_cycle_oscillator(weekly, windows["turns_for_mapping"])
envelopes = compute_envelopes(damping)

fig, ax = plot_cycle_oscillator(
    weekly=weekly,
    cycle=cycle,
    turning_points=turns_actual,
    envelopes=envelopes,
    windows=windows,
)

plt.show()


## Resumen numerico

Este resumen imprime los valores centrales del oscilador historico: precio actual, betas descriptivas, periodo pico-pico, tiempo pico-minimo y rangos estimados de minimo/maximo. Las betas se reportan como descripcion del esquema manual de fases, no como prueba estadistica robusta.

In [ ]:
latest = df.iloc[-1]

print("Resumen numerico:")
print(f"Fecha/precio actual: {latest['date'].date()} - ${latest['price']:,.0f}")
print()
print("Betas de amortiguamiento:")
print(f"Beta superior menos amortiguamiento: {damping['beta_peak_low']:.4f} / año")
print(f"Beta superior mas amortiguamiento:   {damping['beta_peak_high']:.4f} / año")
print(f"Beta inferior menos amortiguamiento: {damping['beta_trough_low']:.4f} / año")
print(f"Beta inferior mas amortiguamiento:   {damping['beta_trough_high']:.4f} / año")
print()
print(f"Periodo pico->pico: {windows['pp_mean']:.2f} +/- {windows['pp_std']:.2f} años")
print(f"Tiempo pico->minimo: {windows['pt_mean']:.2f} +/- {windows['pt_std']:.2f} años")
print()
print("Rangos estimados:")
print(
    "Proximo minimo: "
    f"{windows['next_min_early'].strftime('%Y-%m-%d')} a "
    f"{windows['next_min_late'].strftime('%Y-%m-%d')} | "
    f"${windows['min_price_low']:,.0f} - ${windows['min_price_high']:,.0f}"
)
print(
    "Proximo maximo: "
    f"{windows['next_max_early'].strftime('%Y-%m-%d')} a "
    f"{windows['next_max_late'].strftime('%Y-%m-%d')} | "
    f"${windows['max_price_low']:,.0f} - ${windows['max_price_high']:,.0f}"
)


<!-- PHASE3_GENERATED_CELL -->
## Fase 3: Ensemble de escenarios futuros

Esta fase mantiene el oscilador historico y anade un ensemble probabilistico de trayectorias futuras en el espacio de fase del ciclo. El precio se calcula internamente para analizar entradas, targets y riesgo, pero el grafico principal sigue siendo el oscilador.


<!-- PHASE3_GENERATED_CELL -->
### Uso del halving

El modelo no inventa datos on-chain ni proxies derivados del precio. Usa solo la distancia en dias desde el halving mas reciente para ponderar que ciclos historicos son mas comparables al momento actual: los ciclos con una distancia al halving parecida reciben mas peso al muestrear fechas de minimo/maximo y parametros de ciclo.


<!-- PHASE3_GENERATED_CELL -->
## Funciones fase 3

In [ ]:
# PHASE3_GENERATED_CELL
def verify_dataset_contract(raw_df):
    """Confirm that the local dataset only contains daily close data."""
    expected_columns = ["Time", "BTC / USD Denominated Closing Price"]
    columns = list(raw_df.columns)
    dates = pd.to_datetime(raw_df["Time"], errors="coerce")

    forbidden_keywords = [
        "open", "high", "low", "volume", "mvrv", "hash", "sopr",
        "active", "address", "on-chain", "onchain",
    ]
    unexpected = [
        col for col in columns
        if any(keyword in col.lower() for keyword in forbidden_keywords)
    ]

    print("Dataset verification:")
    print(f"Columns: {columns}")
    print(f"Rows: {len(raw_df):,}")
    print(f"Dates: {dates.min().date()} -> {dates.max().date()}")
    print(f"OHLC/on-chain columns found: {unexpected}")

    assert columns == expected_columns, f"Unexpected dataset columns: {columns}"
    if unexpected:
        raise ValueError(
            "The local dataset contains OHLC/on-chain-like columns. "
            "Stop and review the model assumptions before continuing."
        )

    return {
        "columns": columns,
        "rows": len(raw_df),
        "first_date": dates.min(),
        "last_date": dates.max(),
        "unexpected_columns": unexpected,
    }


def _halving_timestamps(include_next=False):
    dates = [pd.Timestamp(date) for date in HALVING_DATES]
    if include_next:
        dates.append(pd.Timestamp(NEXT_HALVING_ESTIMATE))
    return sorted(dates)


def days_since_recent_halving(date, halving_dates=None):
    """Return days elapsed since the most recent known halving."""
    date = pd.Timestamp(date)
    halvings = [pd.Timestamp(d) for d in (halving_dates or HALVING_DATES)]
    previous = [halving for halving in halvings if halving <= date]
    if not previous:
        return np.nan
    return (date - max(previous)).days


def halving_phase_weight(reference_days, candidate_days, scale_days=220):
    """Weight historical cycles by similarity in days since the latest halving."""
    candidate_days = np.asarray(candidate_days, dtype=float)
    reference_days = float(reference_days)
    weights = np.exp(-0.5 * ((candidate_days - reference_days) / scale_days) ** 2)
    if not np.isfinite(weights).all() or weights.sum() <= 0:
        return np.ones_like(candidate_days) / len(candidate_days)
    return weights / weights.sum()


def _weighted_sample_index(rng, weights):
    weights = np.asarray(weights, dtype=float)
    if not np.isfinite(weights).all() or weights.sum() <= 0:
        weights = np.ones_like(weights) / len(weights)
    else:
        weights = weights / weights.sum()
    return int(rng.choice(np.arange(len(weights)), p=weights))


def bridge_noise(n_steps, scale=0.03, rng=None, rho=0.65):
    """Create AR(1)-style bridge noise that starts and ends near zero."""
    rng = rng or np.random.default_rng()
    n_steps = int(n_steps)
    if n_steps <= 1 or scale <= 0:
        return np.zeros(max(n_steps, 0))

    innovations = rng.normal(0, scale, n_steps)
    ar = np.zeros(n_steps)
    for i in range(1, n_steps):
        ar[i] = rho * ar[i - 1] + innovations[i]

    walk = np.cumsum(ar)
    bridge = walk - np.linspace(walk[0], walk[-1], n_steps)
    bridge -= bridge[0]
    max_abs = np.max(np.abs(bridge))
    if max_abs > 1e-12:
        bridge = bridge / max_abs * scale * rng.uniform(0.65, 1.15)
    bridge[-1] = 0.0
    return bridge


def interpolate_scenario_curve(
    anchor_days,
    anchor_values,
    horizon_days,
    rng=None,
    noise_scale=0.0,
    noise_rho=0.65,
    lower_bounds=None,
    upper_bounds=None,
):
    """Interpolate anchors and add bridge noise segment by segment."""
    rng = rng or np.random.default_rng()
    horizon_days = int(horizon_days)
    x = np.arange(horizon_days)

    anchors = (
        pd.DataFrame({"day": anchor_days, "value": anchor_values})
        .drop_duplicates("day", keep="last")
        .sort_values("day")
    )
    anchors["day"] = anchors["day"].clip(0, horizon_days - 1).astype(int)
    base = np.interp(x, anchors["day"], anchors["value"])

    noisy = base.copy()
    days = anchors["day"].to_numpy()
    for start_day, end_day in zip(days[:-1], days[1:]):
        if end_day <= start_day:
            continue
        segment_slice = slice(start_day, end_day + 1)
        noisy[segment_slice] += bridge_noise(
            end_day - start_day + 1,
            scale=noise_scale,
            rng=rng,
            rho=noise_rho,
        )

    if lower_bounds is not None or upper_bounds is not None:
        lower = lower_bounds if lower_bounds is not None else -np.inf
        upper = upper_bounds if upper_bounds is not None else np.inf
        noisy = np.clip(noisy, lower, upper)

    return noisy

def _cycle_training_samples(turning_points):
    """Extract complete peak-trough-next-peak samples for calibration/simulation."""
    peaks = turning_points[turning_points["kind"] == "max"].sort_values("date").reset_index(drop=True)
    troughs = turning_points[
        (turning_points["kind"] == "min") &
        (turning_points["label"] != "Inicio 2013")
    ].sort_values("date").reset_index(drop=True)

    samples = []
    for i in range(len(peaks) - 1):
        peak = peaks.iloc[i]
        next_peak = peaks.iloc[i + 1]
        trough_candidates = troughs[
            (troughs["date"] > peak["date"]) &
            (troughs["date"] < next_peak["date"])
        ]
        if trough_candidates.empty:
            continue

        trough = trough_candidates.iloc[0]
        samples.append({
            "cycle_label": f"{peak['label']} -> {next_peak['label']}",
            "peak_date": peak["date"],
            "trough_date": trough["date"],
            "next_peak_date": next_peak["date"],
            "peak_price": peak["price"],
            "trough_price": trough["price"],
            "next_peak_price": next_peak["price"],
            "peak_phase": peak["phase"],
            "trough_phase": trough["phase"],
            "next_peak_phase": next_peak["phase"],
            "peak_to_trough_days": (trough["date"] - peak["date"]).days,
            "peak_to_peak_days": (next_peak["date"] - peak["date"]).days,
            "drawdown": 1 - trough["price"] / peak["price"],
            "multiplier": next_peak["price"] / trough["price"],
            "days_since_halving_at_peak": days_since_recent_halving(peak["date"]),
        })

    return pd.DataFrame(samples)


def extract_cycle_shape_templates(cycle, turning_points, min_points=8):
    """Extract historical oscillator residuals to reuse as realistic cycle pulse."""
    turns = turning_points.sort_values("date").reset_index(drop=True)
    templates = []

    for i in range(len(turns) - 1):
        start = turns.iloc[i]
        end = turns.iloc[i + 1]
        if start["kind"] == end["kind"]:
            continue

        seg = cycle[
            (cycle["date"] >= start["date"]) &
            (cycle["date"] <= end["date"])
        ].copy()
        if len(seg) < min_points:
            continue

        total_days = max(1, (pd.Timestamp(end["date"]) - pd.Timestamp(start["date"])).days)
        t = (
            (pd.to_datetime(seg["date"]) - pd.Timestamp(start["date"]))
            .dt.days
            .to_numpy()
            / total_days
        )
        t = np.clip(t, 0, 1)

        baseline = start["phase"] + t * (end["phase"] - start["phase"])
        residual = seg["cycle_smooth"].to_numpy() - baseline
        residual = residual - np.interp(t, [t[0], t[-1]], [residual[0], residual[-1]])

        amplitude = max(abs(end["phase"] - start["phase"]), 0.05)
        templates.append({
            "label": f"{start['label']} -> {end['label']}",
            "direction": "up" if end["phase"] > start["phase"] else "down",
            "t": t,
            "residual": residual,
            "amplitude": amplitude,
            "start_date": start["date"],
            "end_date": end["date"],
        })

    if not templates:
        raise ValueError("No historical cycle shape templates could be extracted.")

    return templates


def sample_cycle_template_residual(templates, direction, n_steps, rng, target_amplitude):
    """Sample and scale a historical residual so current scenarios stay damped."""
    candidates = [template for template in templates if template["direction"] == direction]
    if not candidates or n_steps <= 1:
        return np.zeros(max(int(n_steps), 0)), None

    template = candidates[int(rng.integers(0, len(candidates)))]
    x = np.linspace(0, 1, int(n_steps))
    residual = np.interp(x, template["t"], template["residual"])
    residual = residual - np.linspace(residual[0], residual[-1], len(residual))

    amp_ratio = target_amplitude / max(template["amplitude"], 0.05)
    damping_scale = np.clip(amp_ratio, 0.18, 0.72) * rng.uniform(0.45, 0.85)
    return residual * damping_scale, template["label"]


def _envelope_components_for_dates(dates, damping):
    """Return upper/lower damping bands for arbitrary dates."""
    dates = pd.to_datetime(pd.Series(dates))
    peaks = damping["peaks"]
    troughs = damping["troughs"]

    upper_anchor_date = peaks.iloc[0]["date"]
    upper_anchor_amp = abs(peaks.iloc[0]["phase"])
    upper_t = np.array([max(0, year_diff(date, upper_anchor_date)) for date in dates])
    upper_less_damp = upper_anchor_amp * np.exp(-damping["beta_peak_low"] * upper_t)
    upper_more_damp = upper_anchor_amp * np.exp(-damping["beta_peak_high"] * upper_t)

    lower_anchor_date = troughs.iloc[0]["date"]
    lower_anchor_amp = abs(troughs.iloc[0]["phase"])
    lower_t = np.array([max(0, year_diff(date, lower_anchor_date)) for date in dates])
    lower_less_damp = -lower_anchor_amp * np.exp(-damping["beta_trough_low"] * lower_t)
    lower_more_damp = -lower_anchor_amp * np.exp(-damping["beta_trough_high"] * lower_t)

    return {
        "upper_low": np.minimum(upper_less_damp, upper_more_damp),
        "upper_high": np.maximum(upper_less_damp, upper_more_damp),
        "lower_low": np.minimum(lower_less_damp, lower_more_damp),
        "lower_high": np.maximum(lower_less_damp, lower_more_damp),
    }


def simulate_cycle_scenarios(
    df,
    weekly,
    cycle,
    turning_points,
    damping,
    windows,
    n_models=N_MODELS,
    horizon_days=HORIZON_DAYS,
    random_seed=RANDOM_SEED,
    max_attempt_multiplier=45,
):
    """Generate valid future scenarios with diagnostics for rejected paths."""
    rng = np.random.default_rng(random_seed)
    latest = df.iloc[-1]
    current_date = pd.Timestamp(latest["date"])
    current_price = float(latest["price"])
    current_phase = float(cycle.iloc[-1]["cycle_smooth"])

    last_peak = windows["last_peak"]
    last_peak_date = pd.Timestamp(last_peak["date"])
    last_peak_price = float(last_peak["price"])
    days_from_peak_to_current = max(0, (current_date - last_peak_date).days)

    future_dates = pd.date_range(current_date, periods=horizon_days, freq="D")
    envelope_components = _envelope_components_for_dates(future_dates, damping)
    lower_bounds = envelope_components["lower_low"] - ENVELOPE_MARGIN_PHASE
    upper_bounds = envelope_components["upper_high"] + ENVELOPE_MARGIN_PHASE

    training_samples = _cycle_training_samples(turning_points)
    if training_samples.empty:
        raise ValueError("No complete historical cycles available for scenario sampling.")

    shape_templates = extract_cycle_shape_templates(cycle, turning_points)

    reference_halving_days = days_since_recent_halving(last_peak_date)
    weights = halving_phase_weight(
        reference_halving_days,
        training_samples["days_since_halving_at_peak"].to_numpy(),
    )

    phase_paths = []
    price_paths = []
    meta_rows = []
    attempts = 0
    max_attempts = int(n_models * max_attempt_multiplier)
    rejection_counts = {}

    def reject(reason):
        rejection_counts[reason] = rejection_counts.get(reason, 0) + 1

    while len(phase_paths) < n_models and attempts < max_attempts:
        attempts += 1

        pt_sample = training_samples.iloc[_weighted_sample_index(rng, weights)]
        pp_sample = training_samples.iloc[_weighted_sample_index(rng, weights)]

        pt_days_from_peak = rng.normal(
            pt_sample["peak_to_trough_days"],
            max(14, windows["pt_std"] * 365.25 * 0.50),
        )
        pp_days_from_peak = rng.normal(
            pp_sample["peak_to_peak_days"],
            max(21, windows["pp_std"] * 365.25 * 0.50),
        )

        min_day = int(round(pt_days_from_peak - days_from_peak_to_current))
        max_day = int(round(pp_days_from_peak - days_from_peak_to_current))

        if min_day <= 18 or max_day <= min_day + 60 or max_day >= horizon_days - 8:
            reject("incoherent_timing")
            continue

        min_date = future_dates[min_day]
        max_date = future_dates[max_day]

        drawdown = float(np.clip(
            rng.normal(DRAWDOWN_MEAN, DRAWDOWN_STD),
            DRAWDOWN_MIN,
            DRAWDOWN_MAX,
        ))
        min_price = last_peak_price * (1 - drawdown)

        multiplier = float(np.clip(
            rng.normal(MULT_MEAN, MULT_STD),
            MULT_MIN_CLIP,
            MULT_MAX_CLIP,
        ))
        max_price = min_price * multiplier

        if min_price < MIN_REASONABLE_PRICE:
            reject("min_price_below_floor")
            continue
        if max_price > MAX_REASONABLE_PRICE:
            reject("max_price_above_cap")
            continue
        if max_price < current_price * 1.15:
            reject("max_price_too_low_vs_current")
            continue
        if min_price > current_price * 0.95:
            reject("min_price_too_high_vs_current")
            continue

        min_phase_low = envelope_components["lower_low"][min_day]
        min_phase_high = envelope_components["lower_high"][min_day]
        max_phase_low = envelope_components["upper_low"][max_day]
        max_phase_high = envelope_components["upper_high"][max_day]

        min_phase = float(np.clip(
            rng.uniform(min_phase_low, min_phase_high) + rng.normal(0, 0.018),
            min_phase_low,
            min_phase_high,
        ))
        max_phase = float(np.clip(
            rng.uniform(max_phase_low, max_phase_high) + rng.normal(0, 0.018),
            max_phase_low,
            max_phase_high,
        ))

        terminal_phase = float(np.clip(
            max_phase * rng.uniform(0.45, 0.88) + rng.normal(0, 0.025),
            lower_bounds[-1],
            upper_bounds[-1],
        ))
        terminal_price = float(np.clip(
            max_price * rng.uniform(0.66, 1.08),
            MIN_REASONABLE_PRICE,
            MAX_REASONABLE_PRICE,
        ))

        anchor_days = [0, min_day, max_day, horizon_days - 1]
        phase_anchors = [current_phase, min_phase, max_phase, terminal_phase]
        log_price_anchors = [
            np.log(current_price),
            np.log(min_price),
            np.log(max_price),
            np.log(terminal_price),
        ]

        def add_anchor(day, phase_value, log_price_value):
            day = int(np.clip(day, 1, horizon_days - 2))
            anchor_days.append(day)
            phase_anchors.append(float(phase_value))
            log_price_anchors.append(float(log_price_value))

        down_amp = max(abs(current_phase - min_phase), 0.05)
        down_candidates = [
            (0.16, 0.08, 0.80),
            (0.36, -0.05, 0.86),
            (0.58, 0.06, 0.76),
            (0.82, -0.08, 0.88),
        ]
        for frac, price_offset, probability in down_candidates:
            if rng.random() > probability:
                continue
            jitter = rng.normal(0, max(2, min_day * 0.035))
            day = int(np.clip(frac * min_day + jitter, 1, min_day - 1))
            base_phase = current_phase + frac * (min_phase - current_phase)
            base_log_price = np.log(current_price) + frac * (np.log(min_price) - np.log(current_price))
            phase_offset = rng.choice([1, -1]) * rng.uniform(0.05, 0.20)
            add_anchor(
                day,
                base_phase + phase_offset * down_amp,
                base_log_price + price_offset * rng.uniform(0.35, 0.85),
            )

        up_amp = max(abs(max_phase - min_phase), 0.05)
        up_span = max_day - min_day
        up_candidates = [
            (0.16, 0.10, 0.10, 0.90),
            (0.34, -0.20, -0.12, 0.92),
            (0.52, -0.06, -0.03, 0.72),
            (0.70, 0.10, 0.08, 0.86),
            (0.88, 0.16, 0.11, 0.78),
        ]
        for frac, phase_offset, price_offset, probability in up_candidates:
            if rng.random() > probability:
                continue
            jitter = rng.normal(0, max(3, up_span * 0.035))
            day = int(np.clip(min_day + frac * up_span + jitter, min_day + 1, max_day - 1))
            base_phase = min_phase + frac * (max_phase - min_phase)
            base_log_price = np.log(min_price) + frac * (np.log(max_price) - np.log(min_price))
            add_anchor(
                day,
                base_phase + phase_offset * up_amp * rng.uniform(0.60, 1.10),
                base_log_price + price_offset * rng.uniform(0.60, 1.15),
            )

        post_span = horizon_days - 1 - max_day
        if post_span > 90:
            post_candidates = [
                (0.18, max_phase * rng.uniform(0.78, 0.92), np.log(max_price * rng.uniform(0.82, 0.96)), 0.82),
                (0.48, terminal_phase + rng.normal(0, 0.035), np.log(terminal_price * rng.uniform(0.92, 1.06)), 0.70),
            ]
            for frac, phase_value, log_price_value, probability in post_candidates:
                if rng.random() > probability:
                    continue
                jitter = rng.normal(0, max(3, post_span * 0.045))
                day = int(np.clip(max_day + frac * post_span + jitter, max_day + 1, horizon_days - 2))
                add_anchor(day, phase_value, log_price_value)

        phase_noise_scale = rng.uniform(PHASE_NOISE_SCALE_MIN, PHASE_NOISE_SCALE_MAX)
        phase_noise_rho = rng.uniform(PHASE_NOISE_RHO_MIN, PHASE_NOISE_RHO_MAX)
        price_noise_scale = rng.uniform(PRICE_NOISE_SCALE_MIN, PRICE_NOISE_SCALE_MAX)

        raw_phase_path = interpolate_scenario_curve(
            anchor_days,
            phase_anchors,
            horizon_days,
            rng=rng,
            noise_scale=phase_noise_scale,
            noise_rho=phase_noise_rho,
        )

        down_residual, down_template = sample_cycle_template_residual(
            shape_templates,
            "down",
            min_day + 1,
            rng,
            target_amplitude=down_amp,
        )
        up_residual, up_template = sample_cycle_template_residual(
            shape_templates,
            "up",
            up_span + 1,
            rng,
            target_amplitude=up_amp,
        )
        raw_phase_path[:min_day + 1] += down_residual
        raw_phase_path[min_day:max_day + 1] += up_residual

        overshoot = max(
            float(np.max(raw_phase_path - upper_bounds)),
            float(np.max(lower_bounds - raw_phase_path)),
            0.0,
        )
        if overshoot > ENVELOPE_MARGIN_PHASE * 2.2:
            reject("phase_outside_envelope")
            continue
        phase_path = np.clip(raw_phase_path, lower_bounds, upper_bounds)

        log_price_path = interpolate_scenario_curve(
            anchor_days,
            log_price_anchors,
            horizon_days,
            rng=rng,
            noise_scale=price_noise_scale,
            noise_rho=phase_noise_rho,
        )
        price_path = np.exp(log_price_path)

        if price_path.min() < MIN_REASONABLE_PRICE:
            reject("path_min_price_below_floor")
            continue
        if price_path.max() > MAX_REASONABLE_PRICE:
            reject("path_max_price_above_cap")
            continue
        if not np.isfinite(phase_path).all() or not np.isfinite(price_path).all():
            reject("non_finite_path")
            continue

        phase_paths.append(phase_path)
        price_paths.append(price_path)
        meta_rows.append({
            "scenario_id": len(meta_rows),
            "min_day": min_day,
            "max_day": max_day,
            "next_min_date": min_date,
            "next_max_date": max_date,
            "next_min_price": min_price,
            "next_max_price": max_price,
            "drawdown": drawdown,
            "multiplier": multiplier,
            "min_phase": min_phase,
            "max_phase": max_phase,
            "n_internal_anchors": max(0, len(anchor_days) - 4),
            "phase_noise_scale": phase_noise_scale,
            "phase_noise_rho": phase_noise_rho,
            "price_noise_scale": price_noise_scale,
            "template_down": down_template,
            "template_up": up_template,
            "sampled_pt_cycle": pt_sample["cycle_label"],
            "sampled_pp_cycle": pp_sample["cycle_label"],
            "days_since_halving_start": days_since_recent_halving(current_date),
            "days_since_halving_last_peak": reference_halving_days,
            "days_since_halving_min": days_since_recent_halving(min_date),
            "days_since_halving_max": days_since_recent_halving(max_date),
        })

    rejection_summary = (
        pd.Series(rejection_counts, name="rejections")
        .sort_values(ascending=False)
        .rename_axis("reason")
        .reset_index()
    )

    print(f"Scenario generation attempts: {attempts:,}")
    print(f"Valid scenarios generated: {len(meta_rows):,}")
    if len(rejection_summary):
        print("Rejected scenarios by reason:")
        display(rejection_summary)

    if len(phase_paths) < n_models:
        raise RuntimeError(
            f"Only generated {len(phase_paths)} valid scenarios after {attempts} attempts. "
            "Relax filters or reduce n_models."
        )

    phase_paths = np.vstack(phase_paths)
    price_paths = np.vstack(price_paths)
    scenario_meta = pd.DataFrame(meta_rows)
    scenario_meta.attrs["rejection_summary"] = rejection_summary

    print(f"phase_paths shape: {phase_paths.shape}")
    print(f"price_paths shape: {price_paths.shape}")

    return phase_paths, price_paths, scenario_meta, future_dates

def wilson_interval(successes, n, z=1.96):
    """Wilson confidence interval for a binomial proportion."""
    if n <= 0:
        return np.nan, np.nan
    p = successes / n
    denom = 1 + z ** 2 / n
    center = (p + z ** 2 / (2 * n)) / denom
    margin = z * np.sqrt((p * (1 - p) + z ** 2 / (4 * n)) / n) / denom
    return max(0, center - margin), min(1, center + margin)


def backtest_walk_forward_cycles(
    turning_points,
    n_models=BACKTEST_N_MODELS,
    random_seed=RANDOM_SEED,
):
    """Leave one historical cycle out and test calibration without global dispersion leakage."""
    rng = np.random.default_rng(random_seed)
    samples = _cycle_training_samples(turning_points)
    if len(samples) < 3:
        raise ValueError("Need at least three complete cycles for leave-one-cycle-out backtesting.")

    drawdown_thresholds = np.array([0.35, 0.45, 0.55, 0.65, 0.75])
    multiplier_thresholds = np.array([2.0, 3.0, 4.0, 5.0, 6.0])
    events = []
    fold_rows = []

    for validation_idx in range(len(samples)):
        validation = samples.iloc[validation_idx]
        train = samples.drop(validation_idx).reset_index(drop=True)
        weights = halving_phase_weight(
            validation["days_since_halving_at_peak"],
            train["days_since_halving_at_peak"].to_numpy(),
        )

        train_drawdowns = train["drawdown"].to_numpy(dtype=float)
        train_multipliers = train["multiplier"].to_numpy(dtype=float)

        fold_drawdown_std = max(float(np.std(train_drawdowns, ddof=1)), 0.03)
        fold_multiplier_std = max(float(np.std(train_multipliers, ddof=1)), 0.35)
        fold_drawdown_low = max(0.05, float(train_drawdowns.min() - fold_drawdown_std))
        fold_drawdown_high = min(0.95, float(train_drawdowns.max() + fold_drawdown_std))
        fold_multiplier_low = max(1.05, float(train_multipliers.min() - fold_multiplier_std))
        fold_multiplier_high = max(fold_multiplier_low + 0.25, float(train_multipliers.max() + fold_multiplier_std))

        fold_rows.append({
            "validation_cycle": validation["cycle_label"],
            "drawdown_std_train": fold_drawdown_std,
            "drawdown_clip_low_train": fold_drawdown_low,
            "drawdown_clip_high_train": fold_drawdown_high,
            "multiplier_std_train": fold_multiplier_std,
            "multiplier_clip_low_train": fold_multiplier_low,
            "multiplier_clip_high_train": fold_multiplier_high,
        })

        sampled_drawdowns = []
        sampled_multipliers = []
        for _ in range(n_models):
            idx = _weighted_sample_index(rng, weights)
            row = train.iloc[idx]
            sampled_drawdowns.append(float(np.clip(
                rng.normal(row["drawdown"], fold_drawdown_std),
                fold_drawdown_low,
                fold_drawdown_high,
            )))
            sampled_multipliers.append(float(np.clip(
                rng.normal(row["multiplier"], fold_multiplier_std),
                fold_multiplier_low,
                fold_multiplier_high,
            )))

        sampled_drawdowns = np.array(sampled_drawdowns)
        sampled_multipliers = np.array(sampled_multipliers)

        for threshold in drawdown_thresholds:
            predicted = float((sampled_drawdowns >= threshold).mean())
            observed = int(validation["drawdown"] >= threshold)
            events.append({
                "validation_cycle": validation["cycle_label"],
                "event_type": "drawdown",
                "event": f"drawdown >= {threshold:.0%}",
                "predicted_probability": predicted,
                "observed": observed,
            })

        for threshold in multiplier_thresholds:
            predicted = float((sampled_multipliers >= threshold).mean())
            observed = int(validation["multiplier"] >= threshold)
            events.append({
                "validation_cycle": validation["cycle_label"],
                "event_type": "multiplier",
                "event": f"multiplier >= {threshold:.1f}x",
                "predicted_probability": predicted,
                "observed": observed,
            })

    backtest_events = pd.DataFrame(events)
    backtest_events["brier"] = (
        backtest_events["predicted_probability"] - backtest_events["observed"]
    ) ** 2

    calibration_rows = []
    bins = CALIBRATION_BINS
    for i in range(len(bins) - 1):
        low = bins[i]
        high = bins[i + 1]
        if i == len(bins) - 2:
            mask = (
                (backtest_events["predicted_probability"] >= low) &
                (backtest_events["predicted_probability"] <= high)
            )
        else:
            mask = (
                (backtest_events["predicted_probability"] >= low) &
                (backtest_events["predicted_probability"] < high)
            )
        bucket = backtest_events[mask]
        n = len(bucket)
        successes = int(bucket["observed"].sum()) if n else 0
        ci_low, ci_high = wilson_interval(successes, n)
        calibration_rows.append({
            "bin_low": low,
            "bin_high": high,
            "n": n,
            "predicted_mean": bucket["predicted_probability"].mean() if n else np.nan,
            "observed_frequency": successes / n if n else np.nan,
            "ci_low": ci_low,
            "ci_high": ci_high,
        })

    calibration = pd.DataFrame(calibration_rows)
    brier_summary = pd.DataFrame([
        {
            "scope": "all_events",
            "n_events": len(backtest_events),
            "brier_score": backtest_events["brier"].mean(),
        },
        {
            "scope": "drawdown_events",
            "n_events": int((backtest_events["event_type"] == "drawdown").sum()),
            "brier_score": backtest_events.loc[
                backtest_events["event_type"] == "drawdown", "brier"
            ].mean(),
        },
        {
            "scope": "multiplier_events",
            "n_events": int((backtest_events["event_type"] == "multiplier").sum()),
            "brier_score": backtest_events.loc[
                backtest_events["event_type"] == "multiplier", "brier"
            ].mean(),
        },
    ])

    fold_parameters = pd.DataFrame(fold_rows)

    print("Walk-forward leave-one-cycle-out backtest:")
    print(f"Validation cycles: {len(samples)}")
    print(f"Events tested: {len(backtest_events)}")
    print(f"Brier score, all events: {brier_summary.iloc[0]['brier_score']:.4f}")
    print("Calibration note: sample is very small; use this as indicative, not robust.")

    return backtest_events, calibration, brier_summary, fold_parameters

def validate_ensemble_outputs(phase_paths, price_paths, scenario_meta, expected_models=None):
    """Validate ensemble shape, finite values, and non-degenerate percentiles."""
    phase_paths = np.asarray(phase_paths)
    price_paths = np.asarray(price_paths)
    if phase_paths.shape != price_paths.shape:
        raise AssertionError("phase_paths and price_paths shapes differ.")
    if expected_models is not None and len(scenario_meta) != expected_models:
        raise AssertionError(f"Expected {expected_models} scenarios, got {len(scenario_meta)}.")
    if not np.isfinite(phase_paths).all():
        raise AssertionError("phase_paths contains NaN or Inf.")
    if not np.isfinite(price_paths).all():
        raise AssertionError("price_paths contains NaN or Inf.")

    rows = []
    metrics = {
        "next_min_price": scenario_meta["next_min_price"].to_numpy(dtype=float),
        "next_max_price": scenario_meta["next_max_price"].to_numpy(dtype=float),
        "terminal_price": price_paths[:, -1],
    }
    for name, values in metrics.items():
        p05, p50, p95 = np.percentile(values, [5, 50, 95])
        ok = bool(p05 < p50 < p95)
        if len(values) > 1 and not ok:
            warnings.warn(
                f"Degenerate ensemble percentiles for {name}: p05={p05}, p50={p50}, p95={p95}",
                RuntimeWarning,
            )
        rows.append({"metric": name, "p05": p05, "p50": p50, "p95": p95, "strictly_ordered": ok})

    return pd.DataFrame(rows)


def probability_ci_from_calibration(probability, calibration):
    """Map a model probability to the nearest walk-forward calibration interval."""
    probability = float(probability)
    if calibration is None or calibration.empty:
        return 0.0, 1.0, "no_backtest"

    row = calibration[
        (calibration["bin_low"] <= probability) &
        (probability <= calibration["bin_high"])
    ]
    if row.empty:
        row = calibration.iloc[[(calibration["predicted_mean"] - probability).abs().idxmin()]]

    row = row.iloc[0]
    if row["n"] <= 0 or pd.isna(row["ci_low"]) or pd.isna(row["ci_high"]):
        return 0.0, 1.0, "very_wide"

    note = "wide" if (row["ci_high"] - row["ci_low"] > 0.35 or row["n"] < 8) else "ok"
    return float(row["ci_low"]), float(row["ci_high"]), note


def analyze_entry_prices(
    price_paths,
    future_dates,
    current_price,
    entry_prices=ENTRY_PRICES,
    target_prices=TARGET_PRICES,
    calibration=None,
):
    """Estimate entry, miss, target-before-entry, and buy-now-vs-wait probabilities."""
    results = []
    terminal_prices = price_paths[:, -1]
    buy_now_terminal_mult = terminal_prices / current_price

    for entry in entry_prices:
        entry = float(entry)
        touch_matrix = price_paths <= entry
        touched = touch_matrix.any(axis=1)
        touch_idx = np.where(touched, touch_matrix.argmax(axis=1), -1)

        p_touch = float(touched.mean())
        p_miss = 1 - p_touch
        days_to_touch = touch_idx[touched]

        drop_20 = []
        drop_30 = []
        double_after = []
        for scenario_idx, idx in enumerate(touch_idx):
            if idx < 0:
                continue
            after = price_paths[scenario_idx, idx:]
            drop_20.append(after.min() <= entry * 0.80)
            drop_30.append(after.min() <= entry * 0.70)
            double_after.append(after.max() >= entry * 2.0)

        wait_terminal_mult = np.where(touched, terminal_prices / entry, 1.0)

        row = {
            "entry_price": entry,
            "p_toca_entry": p_touch,
            "p_quedarse_fuera": p_miss,
            "dias_medianos_hasta_compra": np.median(days_to_touch) if len(days_to_touch) else np.nan,
            "dias_medios_hasta_compra": np.mean(days_to_touch) if len(days_to_touch) else np.nan,
            "p_caida_20_despues_de_comprar": np.mean(drop_20) if drop_20 else np.nan,
            "p_caida_30_despues_de_comprar": np.mean(drop_30) if drop_30 else np.nan,
            "p_doblar_despues_de_comprar": np.mean(double_after) if double_after else np.nan,
            "valor_esperado_esperar": wait_terminal_mult.mean(),
            "valor_mediano_esperar": np.median(wait_terminal_mult),
            "valor_esperado_comprar_hoy": buy_now_terminal_mult.mean(),
            "valor_mediano_comprar_hoy": np.median(buy_now_terminal_mult),
        }

        for target in target_prices:
            target = float(target)
            target_matrix = price_paths >= target
            target_touched = target_matrix.any(axis=1)
            target_idx = np.where(target_touched, target_matrix.argmax(axis=1), -1)
            before_entry = target_touched & ((~touched) | (target_idx < touch_idx))
            row[f"p_llega_{int(target)}_antes_de_entry"] = float(before_entry.mean())

        probability_columns = [
            "p_toca_entry",
            "p_quedarse_fuera",
            "p_caida_20_despues_de_comprar",
            "p_caida_30_despues_de_comprar",
            "p_doblar_despues_de_comprar",
        ] + [f"p_llega_{int(target)}_antes_de_entry" for target in target_prices]

        for col in probability_columns:
            value = row[col]
            if pd.isna(value):
                row[f"{col}_ci_low"] = np.nan
                row[f"{col}_ci_high"] = np.nan
                row[f"{col}_ci_note"] = "not_applicable"
            else:
                ci_low, ci_high, note = probability_ci_from_calibration(value, calibration)
                row[f"{col}_ci_low"] = ci_low
                row[f"{col}_ci_high"] = ci_high
                row[f"{col}_ci_note"] = note

        results.append(row)

    return pd.DataFrame(results)


def _format_probability_with_ci(row, column):
    value = row[column]
    low = row.get(f"{column}_ci_low", np.nan)
    high = row.get(f"{column}_ci_high", np.nan)
    note = row.get(f"{column}_ci_note", "")

    if pd.isna(value):
        return "n/a"
    if pd.isna(low) or pd.isna(high):
        return f"{value * 100:.1f}%"

    label = f"{value * 100:.1f}% [{low * 100:.1f}, {high * 100:.1f}]"
    if note in {"wide", "very_wide"}:
        label += " wide"
    return label


def format_entry_analysis_table(entry_analysis):
    """Format the final decision table with percentages and calibration intervals."""
    formatted = pd.DataFrame()
    formatted["entry_price"] = entry_analysis["entry_price"].map(lambda x: f"${x:,.0f}")

    probability_columns = [
        "p_toca_entry",
        "p_quedarse_fuera",
        "p_caida_20_despues_de_comprar",
        "p_caida_30_despues_de_comprar",
        "p_doblar_despues_de_comprar",
        "p_llega_100000_antes_de_entry",
        "p_llega_150000_antes_de_entry",
        "p_llega_200000_antes_de_entry",
        "p_llega_250000_antes_de_entry",
    ]
    for col in probability_columns:
        if col in entry_analysis:
            formatted[col] = entry_analysis.apply(lambda row: _format_probability_with_ci(row, col), axis=1)

    multiplier_columns = [
        "valor_esperado_esperar",
        "valor_mediano_esperar",
        "valor_esperado_comprar_hoy",
        "valor_mediano_comprar_hoy",
    ]
    for col in multiplier_columns:
        formatted[col] = entry_analysis[col].map(lambda x: f"{x:.2f}x")

    formatted["dias_medianos_hasta_compra"] = entry_analysis["dias_medianos_hasta_compra"].round(0)
    formatted["dias_medios_hasta_compra"] = entry_analysis["dias_medios_hasta_compra"].round(0)
    return formatted


def plot_calibration_diagram(calibration, brier_summary):
    """Plot predicted probability vs observed frequency from the walk-forward backtest."""
    fig, ax = plt.subplots(figsize=(7, 6))

    valid = calibration.dropna(subset=["predicted_mean", "observed_frequency"])
    ax.plot([0, 1], [0, 1], "--", color="gray", label="Calibracion perfecta")
    if not valid.empty:
        yerr = np.vstack([
            valid["observed_frequency"] - valid["ci_low"],
            valid["ci_high"] - valid["observed_frequency"],
        ])
        ax.errorbar(
            valid["predicted_mean"],
            valid["observed_frequency"],
            yerr=yerr,
            fmt="o",
            color="#3f73b5",
            capsize=4,
            label="Walk-forward observado",
        )
        for _, row in valid.iterrows():
            ax.annotate(
                f"n={int(row['n'])}",
                (row["predicted_mean"], row["observed_frequency"]),
                textcoords="offset points",
                xytext=(6, 6),
                fontsize=8,
            )

    brier = brier_summary.loc[brier_summary["scope"] == "all_events", "brier_score"].iloc[0]
    n_events = int(brier_summary.loc[brier_summary["scope"] == "all_events", "n_events"].iloc[0])
    ax.set_title("Reliability diagram: walk-forward leave-one-cycle-out")
    ax.set_xlabel("Probabilidad predicha")
    ax.set_ylabel("Frecuencia observada")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.grid(True, alpha=0.3)
    ax.legend(loc="upper left")
    ax.text(
        0.03,
        0.04,
        f"Brier: {brier:.3f}\\nEventos: {n_events}\\nMuestra pequena: indicativo, no robusto",
        transform=ax.transAxes,
        fontsize=9,
        bbox=dict(boxstyle="round,pad=0.35", facecolor="white", alpha=0.9),
    )
    plt.tight_layout()
    return fig, ax


def compute_ensemble_extrema_stats(
    phase_paths,
    price_paths,
    future_dates,
    scenario_meta=None,
    price_divergence_threshold=0.05,
    date_divergence_threshold_days=30,
):
    """Compute ensemble trough/peak stats, using design anchors as source of truth when available."""
    phase_paths = np.asarray(phase_paths)
    price_paths = np.asarray(price_paths)
    future_dates = pd.to_datetime(pd.Index(future_dates))

    if phase_paths.shape != price_paths.shape:
        raise ValueError("phase_paths and price_paths must have the same shape.")
    if phase_paths.shape[1] != len(future_dates):
        raise ValueError("future_dates length must match path horizon.")

    def nearest_indices(dates):
        dates = pd.to_datetime(pd.Index(dates))
        return np.array([
            int(np.argmin(np.abs((future_dates - date).days)))
            for date in dates
        ])

    if scenario_meta is not None and {
        "next_min_date", "next_min_price", "next_max_date", "next_max_price"
    }.issubset(scenario_meta.columns):
        per_scenario = scenario_meta[[
            "scenario_id",
            "next_min_date",
            "next_min_price",
            "next_max_date",
            "next_max_price",
        ]].copy()
        min_idx = nearest_indices(per_scenario["next_min_date"])
        max_idx = nearest_indices(per_scenario["next_max_date"])
        scenario_ids = per_scenario["scenario_id"].to_numpy(dtype=int)

        per_scenario["min_idx"] = min_idx
        per_scenario["max_after_min_idx"] = max_idx
        per_scenario["min_date"] = pd.to_datetime(per_scenario["next_min_date"])
        per_scenario["min_price"] = per_scenario["next_min_price"].astype(float)
        per_scenario["min_phase"] = phase_paths[scenario_ids, min_idx]
        per_scenario["max_after_min_date"] = pd.to_datetime(per_scenario["next_max_date"])
        per_scenario["max_after_min_price"] = per_scenario["next_max_price"].astype(float)
        per_scenario["max_after_min_phase"] = phase_paths[scenario_ids, max_idx]
        per_scenario["source"] = "scenario_meta_design_anchor"

        path_min_idx = np.nanargmin(price_paths, axis=1)
        path_min_price = price_paths[np.arange(len(price_paths)), path_min_idx]
        path_max_idx = []
        path_max_price = []
        for scenario_id, idx in enumerate(path_min_idx):
            offset = int(np.nanargmax(price_paths[scenario_id, idx:]))
            max_i = int(idx + offset)
            path_max_idx.append(max_i)
            path_max_price.append(price_paths[scenario_id, max_i])
        path_max_idx = np.array(path_max_idx)
        path_max_price = np.array(path_max_price)

        per_scenario["path_min_price"] = path_min_price[scenario_ids]
        per_scenario["path_min_date"] = future_dates[path_min_idx[scenario_ids]]
        per_scenario["path_max_after_min_price"] = path_max_price[scenario_ids]
        per_scenario["path_max_after_min_date"] = future_dates[path_max_idx[scenario_ids]]
        per_scenario["min_price_divergence"] = (
            (per_scenario["path_min_price"] - per_scenario["min_price"]).abs()
            / per_scenario["min_price"].abs()
        )
        per_scenario["max_price_divergence"] = (
            (per_scenario["path_max_after_min_price"] - per_scenario["max_after_min_price"]).abs()
            / per_scenario["max_after_min_price"].abs()
        )
        per_scenario["min_date_divergence_days"] = (
            pd.to_datetime(per_scenario["path_min_date"]) - per_scenario["min_date"]
        ).abs().dt.days
        per_scenario["max_date_divergence_days"] = (
            pd.to_datetime(per_scenario["path_max_after_min_date"]) - per_scenario["max_after_min_date"]
        ).abs().dt.days

        divergent = per_scenario[
            (per_scenario["min_price_divergence"] > price_divergence_threshold) |
            (per_scenario["max_price_divergence"] > price_divergence_threshold) |
            (per_scenario["min_date_divergence_days"] > date_divergence_threshold_days) |
            (per_scenario["max_date_divergence_days"] > date_divergence_threshold_days)
        ]
        if len(divergent):
            warnings.warn(
                f"{len(divergent)} scenarios have noisy path extrema that diverge from design anchors. "
                "Plot labels use design anchors; inspect per_scenario divergence columns if needed.",
                RuntimeWarning,
            )
    else:
        rows = []
        for scenario_id in range(price_paths.shape[0]):
            prices = price_paths[scenario_id]
            phases = phase_paths[scenario_id]
            min_idx = int(np.nanargmin(prices))
            post_prices = prices[min_idx:]
            max_offset = int(np.nanargmax(post_prices))
            max_idx = min_idx + max_offset
            rows.append({
                "scenario_id": scenario_id,
                "min_idx": min_idx,
                "max_after_min_idx": max_idx,
                "min_date": future_dates[min_idx],
                "min_price": float(prices[min_idx]),
                "min_phase": float(phases[min_idx]),
                "max_after_min_date": future_dates[max_idx],
                "max_after_min_price": float(prices[max_idx]),
                "max_after_min_phase": float(phases[max_idx]),
                "source": "path_argmin_argmax",
            })
        per_scenario = pd.DataFrame(rows)

    def mean_timestamp(series):
        values = pd.to_datetime(series).to_numpy(dtype="datetime64[ns]").astype("int64")
        return pd.Timestamp(int(values.mean()))

    def median_timestamp(series):
        values = pd.to_datetime(series).to_numpy(dtype="datetime64[ns]").astype("int64")
        return pd.Timestamp(int(np.median(values)))

    stats = {
        "per_scenario": per_scenario,
        "min_price_mean": per_scenario["min_price"].mean(),
        "min_price_median": per_scenario["min_price"].median(),
        "min_date_mean": mean_timestamp(per_scenario["min_date"]),
        "min_date_median": median_timestamp(per_scenario["min_date"]),
        "min_phase_median": per_scenario["min_phase"].median(),
        "min_phase_mean": per_scenario["min_phase"].mean(),
        "max_after_min_price_mean": per_scenario["max_after_min_price"].mean(),
        "max_after_min_price_median": per_scenario["max_after_min_price"].median(),
        "max_after_min_date_mean": mean_timestamp(per_scenario["max_after_min_date"]),
        "max_after_min_date_median": median_timestamp(per_scenario["max_after_min_date"]),
        "max_after_min_phase_median": per_scenario["max_after_min_phase"].median(),
        "max_after_min_phase_mean": per_scenario["max_after_min_phase"].mean(),
    }

    return stats

def plot_cycle_ensemble(
    weekly,
    cycle,
    turning_points,
    envelopes,
    windows,
    phase_paths,
    scenario_meta,
    future_dates,
    price_paths=None,
    n_plot=N_PLOT,
    future_end=FUTURE_END,
    random_seed=RANDOM_SEED,
    show_bands=SHOW_ENSEMBLE_BANDS,
    show_median=SHOW_ENSEMBLE_MEDIAN,
    path_alpha=ENSEMBLE_PATH_ALPHA,
    path_linewidth=ENSEMBLE_PATH_LINEWIDTH,
    show_extrema_labels=SHOW_ENSEMBLE_EXTREMA_LABELS,
    extrema_use=ENSEMBLE_EXTREMA_USE,
):
    """Plot historical oscillator plus future individual paths and optional labels."""
    rng = np.random.default_rng(random_seed)
    fig, ax = plt.subplots(figsize=(18, 8))

    cycle_spans = [
        (pd.Timestamp("2013-11-01"), pd.Timestamp("2017-12-31"), "#eaf4ff", "Ciclo 2013-2017"),
        (pd.Timestamp("2017-12-31"), pd.Timestamp("2021-11-30"), "#fff3e6", "Ciclo 2017-2021"),
        (pd.Timestamp("2021-11-30"), pd.Timestamp("2025-12-31"), "#edf8ee", "Ciclo 2021-2025"),
        (pd.Timestamp("2025-12-31"), future_end, "#f5edff", "Ciclo proyectado 2025-2029"),
    ]
    for start, end, color, _ in cycle_spans:
        ax.axvspan(start, end, color=color, alpha=0.45, zorder=0)

    ax.plot(
        cycle["date"],
        cycle["cycle_smooth"],
        linewidth=2.8,
        color="#3f73b5",
        label="Ciclo BTC normalizado con datos reales",
        zorder=5,
    )
    ax.plot(
        cycle["date"],
        cycle["cycle_raw"],
        linewidth=0.7,
        alpha=0.25,
        color="#f2a65a",
        label="Detalle semanal real",
        zorder=4,
    )
    ax.axhline(0, linewidth=2.5, color="#3f73b5", alpha=0.75, label="Tendencia normalizada")

    env_dates = envelopes["env_dates"]
    ax.fill_between(
        env_dates,
        envelopes["upper_more_damp"],
        envelopes["upper_less_damp"],
        color="#ff7b72",
        alpha=0.13,
        label="Envolvente superior",
        zorder=1,
    )
    ax.fill_between(
        env_dates,
        envelopes["lower_less_damp"],
        envelopes["lower_more_damp"],
        color="#2fbf71",
        alpha=0.13,
        label="Envolvente inferior",
        zorder=1,
    )
    ax.plot(env_dates, envelopes["upper_less_damp"], "--", color="#ff7b72", linewidth=1.8)
    ax.plot(env_dates, envelopes["upper_more_damp"], "--", color="#ff7b72", linewidth=1.8)
    ax.plot(env_dates, envelopes["lower_less_damp"], "--", color="#2fbf71", linewidth=1.8)
    ax.plot(env_dates, envelopes["lower_more_damp"], "--", color="#2fbf71", linewidth=1.8)

    ax.axvspan(
        windows["next_min_early"],
        windows["next_min_late"],
        color="#f6bd60",
        alpha=0.18,
        label="Ventana proximo minimo",
        zorder=1,
    )
    ax.axvspan(
        windows["next_max_early"],
        windows["next_max_late"],
        color="#c77dff",
        alpha=0.14,
        label="Ventana proximo maximo",
        zorder=1,
    )

    n_paths = min(n_plot, len(phase_paths))
    sample_idx = rng.choice(len(phase_paths), size=n_paths, replace=False)
    for idx in sample_idx:
        ax.plot(
            future_dates,
            phase_paths[idx],
            color="#5e60ce",
            alpha=path_alpha,
            linewidth=path_linewidth,
            zorder=2,
        )

    if show_bands or show_median:
        p05, p25, p50, p75, p95 = np.percentile(phase_paths, [5, 25, 50, 75, 95], axis=0)
        if show_bands:
            ax.fill_between(
                future_dates,
                p05,
                p95,
                color="#5e60ce",
                alpha=0.12,
                label="Banda ensemble 5%-95%",
                zorder=2,
            )
            ax.fill_between(
                future_dates,
                p25,
                p75,
                color="#5e60ce",
                alpha=0.22,
                label="Banda ensemble 25%-75%",
                zorder=3,
            )
        if show_median:
            ax.plot(future_dates, p50, color="#4c1d95", linewidth=2.5, label="Mediana ensemble", zorder=6)

    if show_extrema_labels and price_paths is not None:
        extrema_stats = compute_ensemble_extrema_stats(phase_paths, price_paths, future_dates, scenario_meta=scenario_meta)
        if extrema_use not in {"median", "mean"}:
            raise ValueError('extrema_use must be "median" or "mean".')

        suffix = "median" if extrema_use == "median" else "mean"
        label_name = "Mediana" if extrema_use == "median" else "Media"

        min_date = extrema_stats[f"min_date_{suffix}"]
        min_price = extrema_stats[f"min_price_{suffix}"]
        min_phase = extrema_stats[f"min_phase_{suffix}"]
        max_date = extrema_stats[f"max_after_min_date_{suffix}"]
        max_price = extrema_stats[f"max_after_min_price_{suffix}"]
        max_phase = extrema_stats[f"max_after_min_phase_{suffix}"]

        ax.scatter(
            [min_date],
            [min_phase],
            s=95,
            color="#f6bd60",
            edgecolor="#7c4a03",
            linewidth=1.2,
            zorder=10,
            label="Mínimo ensemble",
        )
        ax.scatter(
            [max_date],
            [max_phase],
            s=95,
            color="#c77dff",
            edgecolor="#5b21b6",
            linewidth=1.2,
            zorder=10,
            label="Máximo ensemble",
        )

        ax.annotate(
            "Mínimo ensemble\n"
            f"{label_name}: ${min_price:,.0f}\n"
            f"Fecha {label_name.lower()}: {min_date.strftime('%b %Y')}",
            (min_date, min_phase),
            textcoords="offset points",
            xytext=(-28, -62),
            ha="right",
            va="top",
            fontsize=9,
            arrowprops=dict(arrowstyle="->", color="#7c4a03", lw=1.0, alpha=0.75),
            bbox=dict(boxstyle="round,pad=0.35", facecolor="white", edgecolor="#f6bd60", alpha=0.86),
            zorder=11,
        )
        ax.annotate(
            "Máximo ensemble\n"
            f"{label_name}: ${max_price:,.0f}\n"
            f"Fecha {label_name.lower()}: {max_date.strftime('%b %Y')}",
            (max_date, max_phase),
            textcoords="offset points",
            xytext=(28, 34),
            ha="left",
            va="bottom",
            fontsize=9,
            arrowprops=dict(arrowstyle="->", color="#5b21b6", lw=1.0, alpha=0.75),
            bbox=dict(boxstyle="round,pad=0.35", facecolor="white", edgecolor="#c77dff", alpha=0.86),
            zorder=11,
        )

    for date in _halving_timestamps(include_next=True):
        style = ":" if date <= pd.Timestamp(NEXT_HALVING_ESTIMATE) else "--"
        ax.axvline(date, color="#111827", linestyle=style, alpha=0.22, linewidth=1.2)
        if date >= pd.Timestamp("2013-01-01"):
            ax.text(date, 1.08, "halving", rotation=90, va="top", ha="right", fontsize=8, alpha=0.55)

    ax.scatter(
        turning_points["date"],
        turning_points["phase"],
        s=55,
        color="#3f73b5",
        zorder=7,
    )
    last_cycle = cycle.iloc[-1]
    ax.scatter(
        last_cycle["date"],
        last_cycle["cycle_smooth"],
        s=95,
        color="#f28e2b",
        zorder=9,
        label="Punto actual",
    )

    ax.set_yticks([-1.0, -0.5, 0.0, 0.5, 1.0])
    ax.set_yticklabels([
        "Capitulacion",
        "Infravalorado",
        "Tendencia",
        "Sobrecalentado",
        "Euforia",
    ])
    ax.set_ylim(-1.15, 1.15)
    ax.set_xlim(pd.Timestamp("2013-01-01"), max(future_end, future_dates[-1]))
    ax.set_title(
        "Bitcoin: ensemble de escenarios futuros en el oscilador de ciclo\n"
        "con ajuste por distancia al halving y pulso historico amortiguado",
        fontsize=15,
    )
    ax.set_xlabel("Fecha")
    ax.set_ylabel("Fase del ciclo")
    ax.grid(True, axis="y", alpha=0.3)
    ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1.0), fontsize=8.5, frameon=True)
    plt.tight_layout(rect=[0, 0, 0.82, 1])
    return fig, ax

def plot_entry_probabilities(entry_analysis, current_price):
    """Plot entry touch, miss, and 150k-before-entry probabilities."""
    fig, ax = plt.subplots(figsize=(12, 6))
    x = entry_analysis["entry_price"]
    ax.plot(x, entry_analysis["p_quedarse_fuera"] * 100, marker="o", label="P(quedarse fuera)")
    ax.plot(x, entry_analysis["p_toca_entry"] * 100, marker="o", label="P(toca mi entrada)")
    col = "p_llega_150000_antes_de_entry"
    if col in entry_analysis:
        ax.plot(x, entry_analysis[col] * 100, marker="o", label="P(llega a 150k antes)")
    ax.axvline(current_price, color="black", linestyle="--", linewidth=1.6, label="Precio actual")
    ax.set_title("Probabilidades segun precio de entrada")
    ax.set_xlabel("Precio de entrada")
    ax.set_ylabel("Probabilidad (%)")
    ax.grid(True, alpha=0.3)
    ax.legend()
    plt.tight_layout()
    return fig, ax


def plot_buy_now_vs_wait(entry_analysis):
    """Compare expected/median terminal multiplier for waiting vs buying now."""
    fig, ax = plt.subplots(figsize=(12, 6))
    x = entry_analysis["entry_price"]
    ax.plot(x, entry_analysis["valor_esperado_esperar"], marker="o", label="Esperar: valor esperado")
    ax.plot(x, entry_analysis["valor_mediano_esperar"], marker="o", label="Esperar: valor mediano")
    ax.plot(x, entry_analysis["valor_esperado_comprar_hoy"], linestyle="--", label="Comprar hoy: valor esperado")
    ax.plot(x, entry_analysis["valor_mediano_comprar_hoy"], linestyle="--", label="Comprar hoy: valor mediano")
    ax.set_title("Comprar hoy vs esperar una entrada")
    ax.set_xlabel("Precio de entrada")
    ax.set_ylabel("Multiplicador terminal")
    ax.grid(True, alpha=0.3)
    ax.legend()
    plt.tight_layout()
    return fig, ax


def summarize_scenarios(scenario_meta, price_paths, phase_paths):
    """Summarize the ensemble distribution for quick review."""
    summary = pd.DataFrame([
        {
            "metric": "Fecha proximo minimo",
            "p05": scenario_meta["next_min_date"].quantile(0.05),
            "p50": scenario_meta["next_min_date"].quantile(0.50),
            "p95": scenario_meta["next_min_date"].quantile(0.95),
        },
        {
            "metric": "Precio proximo minimo",
            "p05": scenario_meta["next_min_price"].quantile(0.05),
            "p50": scenario_meta["next_min_price"].quantile(0.50),
            "p95": scenario_meta["next_min_price"].quantile(0.95),
        },
        {
            "metric": "Fecha proximo maximo",
            "p05": scenario_meta["next_max_date"].quantile(0.05),
            "p50": scenario_meta["next_max_date"].quantile(0.50),
            "p95": scenario_meta["next_max_date"].quantile(0.95),
        },
        {
            "metric": "Precio proximo maximo",
            "p05": scenario_meta["next_max_price"].quantile(0.05),
            "p50": scenario_meta["next_max_price"].quantile(0.50),
            "p95": scenario_meta["next_max_price"].quantile(0.95),
        },
        {
            "metric": "Precio terminal",
            "p05": np.percentile(price_paths[:, -1], 5),
            "p50": np.percentile(price_paths[:, -1], 50),
            "p95": np.percentile(price_paths[:, -1], 95),
        },
        {
            "metric": "Fase terminal",
            "p05": np.percentile(phase_paths[:, -1], 5),
            "p50": np.percentile(phase_paths[:, -1], 50),
            "p95": np.percentile(phase_paths[:, -1], 95),
        },
    ])
    return summary


<!-- PHASE3_GENERATED_CELL -->
## Verificacion del dataset

In [ ]:
# PHASE3_GENERATED_CELL
dataset_check = verify_dataset_contract(raw_df)


<!-- PHASE3_GENERATED_CELL -->
## Simulacion de escenarios

Cada escenario futuro parte de la fase actual, avanza hacia un minimo de ciclo y despues hacia un maximo posterior. No es una extrapolacion suave: se anaden anclas internas, ruido puente y residuales historicos amortiguados. Las probabilidades de entrada se calculan con `price_paths`, aunque el grafico principal se mantenga en fase del oscilador.


In [ ]:
# PHASE3_GENERATED_CELL
phase_paths, price_paths, scenario_meta, future_dates = simulate_cycle_scenarios(
    df=df,
    weekly=weekly,
    cycle=cycle,
    turning_points=turns_actual,
    damping=damping,
    windows=windows,
)

assert phase_paths.shape == (N_MODELS, HORIZON_DAYS)
assert price_paths.shape == (N_MODELS, HORIZON_DAYS)
assert len(scenario_meta) == N_MODELS

ensemble_validation = validate_ensemble_outputs(
    phase_paths,
    price_paths,
    scenario_meta,
    expected_models=N_MODELS,
)
display(ensemble_validation)


<!-- PHASE3_GENERATED_CELL -->
## Backtesting walk-forward

Leave-one-cycle-out significa que se deja un ciclo historico fuera, se calibra con los otros ciclos y se comprueba que probabilidad habria asignado el modelo a eventos del ciclo excluido. Con solo 3 ciclos completos, el Brier score y el reliability diagram son indicativos, no concluyentes.


In [ ]:
# PHASE3_GENERATED_CELL
backtest_events, calibration_df, brier_summary, backtest_fold_parameters = backtest_walk_forward_cycles(
    turns_actual,
    n_models=BACKTEST_N_MODELS,
)

display(brier_summary)
display(calibration_df)
display(backtest_fold_parameters)

fig_calibration, ax_calibration = plot_calibration_diagram(calibration_df, brier_summary)
plt.show()


<!-- PHASE3_GENERATED_CELL -->
## Grafico del ensemble en el oscilador

In [ ]:
# PHASE3_GENERATED_CELL
fig_ensemble, ax_ensemble = plot_cycle_ensemble(
    weekly=weekly,
    cycle=cycle,
    turning_points=turns_actual,
    envelopes=envelopes,
    windows=windows,
    phase_paths=phase_paths,
    price_paths=price_paths,
    scenario_meta=scenario_meta,
    future_dates=future_dates,
)

plt.show()


<!-- PHASE3_GENERATED_CELL -->
## Probabilidades de entrada y tabla final

Esta seccion traduce el ensemble a preguntas de decision: probabilidad de tocar una orden, riesgo de quedarse fuera, targets alcanzados antes de comprar y comparacion entre comprar hoy o esperar. Los intervalos son amplios porque el backtest disponible es pequeno.


In [ ]:
# PHASE3_GENERATED_CELL
current_price = float(df.iloc[-1]["price"])

entry_analysis = analyze_entry_prices(
    price_paths=price_paths,
    future_dates=future_dates,
    current_price=current_price,
    entry_prices=ENTRY_PRICES,
    target_prices=TARGET_PRICES,
    calibration=calibration_df,
)

fig_entries, ax_entries = plot_entry_probabilities(entry_analysis, current_price)
plt.show()

fig_buy_wait, ax_buy_wait = plot_buy_now_vs_wait(entry_analysis)
plt.show()

scenario_summary = summarize_scenarios(scenario_meta, price_paths, phase_paths)
entry_table = format_entry_analysis_table(entry_analysis)

display(scenario_summary)
display(entry_table)


## Limitaciones, correcciones y alcance

**Bugs corregidos en esta version**

- `N_MODELS` queda fijado en `3000`, no en `1`, para que el ensemble tenga distribuciones reales y no probabilidades binarias.
- `N_PLOT` queda en `300`, suficiente para ver nube de trayectorias sin saturar el grafico.
- Las etiquetas de minimo/maximo del ensemble usan ahora las anclas de diseno (`scenario_meta`) como fuente principal de verdad, no el `argmin`/`argmax` del path ruidoso.
- `compute_ensemble_extrema_stats()` mantiene una verificacion cruzada contra los extremos del path y emite warning si ruido y anclas divergen demasiado.
- El backtesting leave-one-cycle-out reestima dispersiones y clips con los ciclos de entrenamiento de cada fold, evitando que los parametros globales contaminen el ciclo held-out.
- Drawdown y multiplicador tienen una sola fuente de verdad: `DRAWDOWN_MEAN/STD` y `MULT_MEAN/STD` derivan los rangos usados por todas las secciones.
- `simulate_cycle_scenarios()` imprime rechazos agrupados por razon para diagnosticar filtros.
- Se valida que no haya NaN/Inf y que los percentiles principales del ensemble no sean degenerados.

**Limitaciones**

- El dataset contiene solo cierre diario BTC/USD. No hay OHLC intradia, volumen, on-chain, derivados, macro ni sentimiento.
- Hay muy pocos ciclos completos comparables; el backtesting es indicativo, no concluyente.
- Los turning points se basan en ventanas manuales. Cambiar esas ventanas puede cambiar de forma material las fases, betas y escenarios.
- La estructura de mercado de Bitcoin ha cambiado con ETFs e institucionales; se asume una continuidad historica que puede no sostenerse.
- Esto no constituye asesoramiento financiero. Es una herramienta paramétrica para razonar bajo incertidumbre.

**Pendiente / fuera de alcance**

- Incorporar datos on-chain, derivados o macro si se quiere una fase futura con variables independientes.
- Sustituir las fases manuales por un ajuste estadistico formal si se quiere eliminar por completo la circularidad del oscilador.
